# set up

In [1]:
#set up
import numpy as np
import pandas as pd

In [4]:
from google.colab import drive
drive.mount('/content/drive')
dic = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/01OrginalData/')
data = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/02DataForAnalysis/')
result = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/03Results/')
brief = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/04DataForR/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load the valence emotion dictionary

In [5]:
Valence_Emotion = pd.read_csv(dic + 'Valence Emotion dictionary.csv')
Valence_Emotion.head()

,Unnamed: 0,Word,V.Mean.Sum,V.SD.Sum,V.Rat.Sum,A.Mean.Sum,A.SD.Sum,A.Rat.Sum,D.Mean.Sum,D.SD.Sum,...,A.Rat.L,A.Mean.H,A.SD.H,A.Rat.H,D.Mean.L,D.SD.L,D.Rat.L,D.Mean.H,D.SD.H,D.Rat.H
0,0,.,0.00,0.00,0,0.00,0.00,0,0.00,0.00,...,0,0.00,0.00,0,0.00,0.00,0,0.00,0.00,0
1,1,aardvark,6.26,2.21,19,2.41,1.40,22,4.27,1.75,...,11,2.55,1.29,11,4.12,1.64,8,4.43,1.99,7
2,2,abalone,5.30,1.59,20,2.65,1.90,20,4.95,1.79,...,12,2.38,1.92,8,5.55,2.21,11,4.36,1.03,11
3,3,abandon,2.84,1.54,19,3.73,2.43,22,3.32,2.50,...,11,3.82,2.14,11,2.77,2.09,13,4.11,2.93,9
4,4,abandonment,2.63,1.74,19,4.95,2.64,21,2.64,1.81,...,14,5.29,2.63,7,2.31,1.45,16,3.08,2.19,12


In [6]:
Valence_Emotion_dict = {}

for i in Valence_Emotion.index:
  Valence_Emotion_dict[Valence_Emotion['Word'][i]] = [Valence_Emotion['V.Mean.Sum'][i],
                                                      Valence_Emotion['A.Mean.Sum'][i],
                                                      Valence_Emotion['D.Mean.Sum'][i]]

# Valence_Emotion_dict.items()

In [7]:
#test
Valence_Emotion_dict['abandon']

[2.84, 3.73, 3.32]

# Load the semantic diversity dictionary

In [8]:
SemD = pd.read_csv(dic + 'SemD.csv')
SemD.head()

,item,mean_cos,SemD,BNC_wordcount,BNC_contexts,BNC_freq,lg_BNC_freq
0,aa,0.020000,1.69,577,314.0,6.6,0.88
1,aah,0.085245,1.07,92,58.0,1.1,0.31
2,aback,0.025864,1.59,294,293.0,3.4,0.64
3,abacus,0.022685,1.64,51,40.0,0.6,0.20
4,abandon,0.008199,2.09,1257,1193.0,14.4,1.19


In [9]:
SemD_dict = {}
for i in SemD.index:
  SemD_dict[SemD['item'][i]] = SemD['SemD'][i]
#SemD_dict.items()

In [10]:
#test
SemD_dict['weird']

1.47

# AD (social media group)

## read the data

In [18]:
#read the data
ad_social = pd.read_csv(result + 'AD_social_lexicon.csv')
ad_social.shape

(352472, 9)

In [14]:
ad_social.head(3)

,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,74.0,377.0,6.366048,20.159151,17.506631,19.628647,0.973684,0.891892,3.083333,3.166667
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,60.0,308.0,4.220779,17.532468,18.506494,19.480519,1.111111,0.950000,4.615385,4.153846
2,2,2024-09-13T01:00:06-07:00,5.042727e+14,False,en,NaN,NaN,NaN,NaN,albums,...,9.0,51.0,1.960784,21.568627,19.607843,17.647059,0.818182,1.111111,9.000000,11.000000


## define a function to retrieve values from disctionaries

In [15]:
# define a function to retrieve values (valuence, arousal, dominance, semd) from the dictionaries**

def get_emotion_values(word):
    valence = Valence_Emotion_dict.get(word, [None, None, None])[0]
    arousal = Valence_Emotion_dict.get(word, [None, None, None])[1]
    dominance = Valence_Emotion_dict.get(word, [None, None, None])[2]
    semd = SemD_dict.get(word, None)
    return pd.Series([valence, arousal, dominance, semd], index=['Valence', 'Arousal', 'Dominance', 'SemD'])





```
输入:

接受一个字符串 word 作为输入（通常是一个单词）。
情感值查找:

Valence_Emotion_dict 是一个字典，其中每个单词对应一个包含 3 个元素的列表（[Valence, Arousal, Dominance]）。
通过 Valence_Emotion_dict.get(word, [None, None, None]) 查找单词的情感值：
Valence: 表示情感的正负性（愉快或不愉快）。
Arousal: 表示情感的唤起度（激动或冷静）。
Dominance: 表示情感的控制力（主导或被支配）。
如果字典中找不到该单词，则返回 [None, None, None]。
语义密度查找:

SemD_dict 是另一个字典，其中每个单词对应一个语义密度值（SemD）。
通过 SemD_dict.get(word, None) 查找单词的语义密度值。
如果字典中没有找到对应的单词，则返回 None。
输出:

将 valence, arousal, dominance, semd 作为 pandas Series 返回。
返回的 Series 的索引为：['Valence', 'Arousal', 'Dominance', 'SemD']。

```





In [19]:
# test
print(get_emotion_values('shocking'))

Valence      4.63
Arousal      5.30
Dominance    4.12
SemD         1.85
dtype: float64


## apply the function to get the value

In [20]:
# Apply the helper function to each row in the DataFrame
ad_social[['Valence', 'Arousal', 'Dominance', 'SemD']] = ad_social['word'].apply(get_emotion_values)

ad_social

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,1.194189e+15,shocking new data has shown that people living...,shocking,['shocking'],1,['shocking'],['ADJ'],['JJ'],4.63,5.30,4.12,1.85
1,0,1.194189e+15,shocking new data has shown that people living...,new,['new'],1,['new'],['ADJ'],['JJ'],7.68,5.14,5.22,2.34
2,0,1.194189e+15,shocking new data has shown that people living...,data,['data'],1,['datum'],['NOUN'],['NNS'],NaN,NaN,NaN,1.70
3,0,1.194189e+15,shocking new data has shown that people living...,has,['has'],1,['have'],['VERB'],['VBZ'],NaN,NaN,NaN,2.34
4,0,1.194189e+15,shocking new data has shown that people living...,shown,['shown'],1,['show'],['VERB'],['VBN'],NaN,NaN,NaN,2.18
...,...,...,...,...,...,...,...,...,...,...,...,...,...
352467,2679,8.267263e+14,young people talk about their experiences of h...,having,['having'],1,['have'],['VERB'],['VBG'],NaN,NaN,NaN,2.26
352468,2679,8.267263e+14,young people talk about their experiences of h...,a,['a'],1,['a'],['PRON'],['DT'],NaN,NaN,NaN,NaN
352469,2679,8.267263e+14,young people talk about their experiences of h...,grandparent,['grandparent'],1,['grandparent'],['NOUN'],['NN'],NaN,NaN,NaN,0.92
352470,2679,8.267263e+14,young people talk about their experiences of h...,with,['with'],1,['with'],['ADP'],['IN'],NaN,NaN,NaN,2.37


In [21]:
# save the results
ad_social.to_csv(result + 'AD_social_lexicon.csv', index=False)
print("The file was successfully saved")

The file was successfully saved


## calcute the mean values for each "id/speaker"

For each unique 'id', calculate the mean (average) for the columns 'Valence', 'Arousal', 'Dominance', and 'SemD'.

In [22]:
df1 = ad_social.groupby(['id']).agg({'Valence': lambda x: np.mean(x),
                                 'Arousal': lambda x: np.mean(x),
                                 'Dominance': lambda x: np.mean(x),
                                  'SemD': lambda x: np.mean(x)}).reset_index() #.reset_index():将 id 从索引恢复为 DataFrame 的普通列
df1.head()

,id,Valence,Arousal,Dominance,SemD
0,2.664801e+14,6.000083,4.040333,5.631000,2.057522
1,2.773841e+14,5.846207,3.950690,5.474828,2.049130
2,2.901395e+14,5.891429,4.203333,5.540476,2.118267
3,2.901494e+14,6.124590,3.991475,5.573770,2.061989
4,2.901739e+14,5.659375,4.125938,5.265000,2.077526


In [23]:
df1.columns

Index(['id', 'Valence', 'Arousal', 'Dominance', 'SemD'], dtype='object')

In [24]:
ad_social_speakers = pd.read_csv(result + 'AD_social_result.csv',engine='python')
ad_social_speakers.head(3)

,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,74.0,377.0,6.366048,20.159151,17.506631,19.628647,0.973684,0.891892,3.083333,3.166667
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,60.0,308.0,4.220779,17.532468,18.506494,19.480519,1.111111,0.950000,4.615385,4.153846
2,2,2024-09-13T01:00:06-07:00,5.042727e+14,False,en,NaN,NaN,NaN,NaN,albums,...,9.0,51.0,1.960784,21.568627,19.607843,17.647059,0.818182,1.111111,9.000000,11.000000


In [25]:
# save the result to the orginal csv. file
# pd.merge(): Merges the two DataFrames on the common column 'id'
# how='left': Ensures that all rows from ad_social_speakers are kept, even if there’s no matching row in pos_result.
merged_df1 = pd.merge(ad_social_speakers, df1[['id','Valence', 'Arousal', 'Dominance', 'SemD']],
                     on='id',
                     how='left')
merged_df1

,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj,Valence,Arousal,Dominance,SemD
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,17.506631,19.628647,0.973684,0.891892,3.083333,3.166667,5.357845,4.075690,5.167586,2.023206
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,18.506494,19.480519,1.111111,0.950000,4.615385,4.153846,5.741923,4.085897,5.452179,2.082808
2,2,2024-09-13T01:00:06-07:00,5.042727e+14,False,en,NaN,NaN,NaN,NaN,albums,...,19.607843,17.647059,0.818182,1.111111,9.000000,11.000000,5.434444,3.875000,5.206111,2.072245
3,3,2024-09-12T01:00:01-07:00,1.286549e+15,False,en,NaN,NaN,NaN,NaN,photos,...,21.186441,17.457627,0.953704,1.213592,3.678571,3.857143,5.761768,4.230061,5.457195,2.033026
4,4,2024-09-09T05:00:01-07:00,1.231950e+15,False,en,NaN,NaN,NaN,NaN,photos,...,22.641509,11.006289,0.636364,2.057143,1.750000,2.750000,5.549048,4.154167,5.484643,2.133972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2675,2675,2010-04-15T07:17:43-07:00,5.187244e+14,False,NaN,alzheimers.org.uk,Alzheimer's Society called on our political pa...,http://www.alzheimers.org.uk/site/scripts/docu...,What are the three main political parties’ ele...,links,...,16.666667,12.500000,0.500000,1.333333,3.000000,6.000000,5.507000,4.185000,4.832000,2.048182
2676,2676,2010-04-01T07:54:09-07:00,3.725374e+15,False,NaN,alzheimers.org.uk,NaN,http://www.alzheimers.org.uk/site/scripts/docu...,www.alzheimers.org.uk,links,...,17.647059,11.764706,0.400000,1.500000,2.000000,5.000000,5.474000,4.285000,5.271000,2.094643
2677,2677,2010-03-31T08:29:41-07:00,8.861507e+14,False,NaN,forum.alzheimers.org.uk,"Many Happy Returns, TP Support for people with...",http://forum.alzheimers.org.uk/showthread.php?...,"Many Happy Returns, TP - Talking Point",links,...,10.000000,40.000000,2.000000,0.250000,4.000000,2.000000,4.563333,4.270000,5.323333,1.961111
2678,2678,2009-09-21T04:18:01-07:00,1.517941e+15,False,NaN,NaN,NaN,NaN,NaN,photos,...,26.923077,30.769231,2.000000,0.875000,8.000000,4.000000,6.013750,4.267500,5.261250,2.071200


In [26]:
# save the results
merged_df1.to_csv(brief + 'AD_social_result.csv', index=False)

# Health (social media group)

In [27]:
#read the data
health_social = pd.read_csv(result + 'Health_social_lexicon.csv')
health_social.shape

(102628, 9)

In [28]:
health_social.head(3)

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,aarp,['aarp'],1,['aarp'],['NOUN'],['NN']
1,0,1.087023e+15,aarp volunteers gathered with user experience ...,volunteers,['volunteers'],1,['volunteer'],['NOUN'],['NNS']
2,0,1.087023e+15,aarp volunteers gathered with user experience ...,gathered,['gathered'],1,['gather'],['VERB'],['VBD']


In [29]:
# Apply the helper function to each row in the DataFrame
health_social[['Valence', 'Arousal', 'Dominance', 'SemD']] = health_social['word'].apply(get_emotion_values)

health_social

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,aarp,['aarp'],1,['aarp'],['NOUN'],['NN'],NaN,NaN,NaN,NaN
1,0,1.087023e+15,aarp volunteers gathered with user experience ...,volunteers,['volunteers'],1,['volunteer'],['NOUN'],['NNS'],NaN,NaN,NaN,1.66
2,0,1.087023e+15,aarp volunteers gathered with user experience ...,gathered,['gathered'],1,['gather'],['VERB'],['VBD'],NaN,NaN,NaN,1.98
3,0,1.087023e+15,aarp volunteers gathered with user experience ...,with,['with'],1,['with'],['ADP'],['IN'],NaN,NaN,NaN,2.37
4,0,1.087023e+15,aarp volunteers gathered with user experience ...,user,['user'],1,['user'],['NOUN'],['NN'],3.67,3.21,5.56,1.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...
102623,3072,8.929571e+15,you may have heard that the federal government...,and,['and'],1,['and'],['CCONJ'],['CC'],NaN,NaN,NaN,2.37
102624,3072,8.929571e+15,you may have heard that the federal government...,how,['how'],1,['how'],['SCONJ'],['WRB'],NaN,NaN,NaN,2.18
102625,3072,8.929571e+15,you may have heard that the federal government...,to,['to'],1,['to'],['PART'],['TO'],NaN,NaN,NaN,2.37
102626,3072,8.929571e+15,you may have heard that the federal government...,get,['get'],1,['get'],['VERB'],['VB'],6.09,3.67,5.65,2.05


In [30]:
# save the results
health_social.to_csv(result + 'Health_social_lexicon.csv', index=False)
print("The file was successfully saved")

The file was successfully saved


In [31]:
df2 = health_social.groupby(['id']).agg({'Valence': lambda x: np.mean(x),
                                 'Arousal': lambda x: np.mean(x),
                                 'Dominance': lambda x: np.mean(x),
                                  'SemD': lambda x: np.mean(x)}).reset_index()
df2.head()

,id,Valence,Arousal,Dominance,SemD
0,1.892159e+14,6.234545,4.825455,6.088182,1.923077
1,2.290554e+14,5.584286,4.104286,5.384286,1.880625
2,2.656471e+14,5.835333,3.968000,5.552667,2.037714
3,2.813105e+14,6.480435,4.710435,6.076087,2.093478
4,2.996404e+14,5.881739,4.077391,5.465652,2.014828


In [32]:
#read the file
health_social_speakers = pd.read_csv(result + 'Health_social_result.csv',engine='python')
health_social_speakers.head(3)


,idx,statistics.like_count,link_attachment.caption,statistics.love_count,post_owner.username,post_owner.name,modified_time,statistics.haha_count,surface.id,lang,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,0,67.0,aarp.org,0.0,AARP,AARP,2024-10-02T10:48:06-07:00,0.0,6.971043e+14,en,...,14.0,47.0,6.382979,19.148936,4.255319,29.787234,1.555556,0.142857,4.666667,3.000000
1,1,96.0,aarp.org,3.0,AARP,AARP,2024-10-02T10:53:20-07:00,2.0,6.971043e+14,en,...,27.0,101.0,5.940594,16.831683,16.831683,26.732673,1.588235,0.629630,4.500000,2.833333
2,2,488.0,NaN,51.0,AARP,AARP,2024-10-02T10:05:42-07:00,2.0,6.971043e+14,en,...,18.0,59.0,5.084746,18.644068,18.644068,30.508475,1.636364,0.611111,6.000000,3.666667


In [33]:
# save the result to the original csv. file
merged_df2 = pd.merge(health_social_speakers, df2[['id','Valence', 'Arousal', 'Dominance', 'SemD']],
                     on='id',
                     how='left')
merged_df2

,idx,statistics.like_count,link_attachment.caption,statistics.love_count,post_owner.username,post_owner.name,modified_time,statistics.haha_count,surface.id,lang,...,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj,Valence,Arousal,Dominance,SemD
0,0,67.0,aarp.org,0.0,AARP,AARP,2024-10-02T10:48:06-07:00,0.0,6.971043e+14,en,...,4.255319,29.787234,1.555556,0.142857,4.666667,3.000000,5.848182,4.050909,5.983636,2.083500
1,1,96.0,aarp.org,3.0,AARP,AARP,2024-10-02T10:53:20-07:00,2.0,6.971043e+14,en,...,16.831683,26.732673,1.588235,0.629630,4.500000,2.833333,5.914688,4.373125,5.705625,2.029651
2,2,488.0,NaN,51.0,AARP,AARP,2024-10-02T10:05:42-07:00,2.0,6.971043e+14,en,...,18.644068,30.508475,1.636364,0.611111,6.000000,3.666667,6.098333,4.405000,5.911111,2.060588
3,3,113.0,NaN,19.0,AARP,AARP,2024-10-01T06:34:55-07:00,0.0,6.971043e+14,en,...,22.580645,29.032258,2.250000,0.777778,9.000000,4.000000,5.968889,3.922222,5.921111,2.128800
4,4,318.0,NaN,74.0,AARP,AARP,2024-09-24T13:01:49-07:00,1.0,6.971043e+14,en,...,22.535211,22.535211,1.333333,1.000000,2.285714,1.714286,6.199500,4.461500,5.818500,2.028852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3068,3068,1.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-29T19:03:18-07:00,0.0,1.180081e+15,en,...,2.380952,21.428571,1.285714,0.111111,1.800000,1.400000,5.384667,4.042667,5.253333,2.042162
3069,3069,0.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-28T05:46:32-07:00,0.0,1.180081e+15,en,...,11.363636,22.727273,1.666667,0.500000,1.666667,1.000000,5.303333,3.702222,5.476667,2.067188
3070,3070,0.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-28T05:45:39-07:00,0.0,1.180081e+15,en,...,10.416667,35.416667,3.400000,0.294118,8.500000,2.500000,5.691538,3.927692,5.635385,2.047750
3071,3071,2.0,NaN,0.0,asaging,American Society on Aging,2024-09-07T06:58:41-07:00,0.0,1.180081e+15,en,...,0.000000,50.000000,5.000000,0.000000,5.000000,1.000000,5.706667,4.996667,6.073333,1.790000


In [34]:
# save the results
merged_df2.to_csv(brief + 'Health_social_result.csv', index=False)

# AD (clinical label group)

In [35]:
#read the data
ad_clinical= pd.read_csv(result + 'AD_clinical_lexicon.csv')
ad_clinical.shape

(62904, 9)

In [36]:
ad_clinical.head(3)

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,mhm,['mhm'],1,['mhm'],['INTJ'],['UH']
1,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,a,['a'],1,['a'],['PRON'],['DT']
2,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,young,['young'],1,['young'],['ADJ'],['JJ']


In [37]:
# Apply the helper function to each row in the DataFrame
ad_clinical[['Valence', 'Arousal', 'Dominance', 'SemD']] = ad_clinical['word'].apply(get_emotion_values)

# Display the DataFrame
ad_clinical

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,mhm,['mhm'],1,['mhm'],['INTJ'],['UH'],NaN,NaN,NaN,NaN
1,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,a,['a'],1,['a'],['PRON'],['DT'],NaN,NaN,NaN,NaN
2,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,young,['young'],1,['young'],['ADJ'],['JJ'],6.31,4.09,5.60,2.00
3,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,boy,['boy'],1,['boy'],['PROPN'],['NNP'],5.84,4.11,5.50,1.61
4,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,going,['going'],1,['go'],['VERB'],['VBG'],NaN,NaN,NaN,2.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
62899,194,Baycrest8961,mhm. yeah. um yes I can remember at least when...,was,['was'],1,['be'],['AUX'],['VBD'],NaN,NaN,NaN,2.33
62900,194,Baycrest8961,mhm. yeah. um yes I can remember at least when...,caused,['caused'],1,['cause'],['VERB'],['VBD'],NaN,NaN,NaN,2.14
62901,194,Baycrest8961,mhm. yeah. um yes I can remember at least when...,by,['by'],1,['by'],['ADP'],['IN'],NaN,NaN,NaN,2.38
62902,194,Baycrest8961,mhm. yeah. um yes I can remember at least when...,long,['long'],1,['long'],['ADV'],['RB'],NaN,NaN,NaN,2.27


In [38]:
# save the results
ad_clinical.to_csv(result + 'AD_clinical_lexicon.csv', index=False)
print("The file was successfully saved")

The file was successfully saved


In [39]:
df3 = ad_clinical.groupby(['id']).agg({'Valence': lambda x: np.mean(x),
                                 'Arousal': lambda x: np.mean(x),
                                 'Dominance': lambda x: np.mean(x),
                                  'SemD': lambda x: np.mean(x)}).reset_index()
df3.head()

,id,Valence,Arousal,Dominance,SemD
0,01-1,6.029741,3.963707,5.664052,2.138268
1,05-1,6.087375,4.155250,5.439375,2.052599
2,06-1,6.180524,3.981152,5.735288,2.136862
3,09-1,6.161955,3.981729,5.682331,2.130751
4,10-1,6.010794,3.905374,5.669860,2.133612


In [40]:
ad_clinical_speakers = pd.read_csv(result + 'AD_clinical_result.csv',engine='python')
ad_clinical_speakers.head(3)

,idx,id,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...,0.029,0.951,0.020,...,10,99,2.020202,18.181818,25.252525,10.101010,0.555556,2.500000,5.000000,9.000000
1,1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...,0.013,0.916,0.071,...,23,179,1.675978,10.614525,30.167598,12.849162,1.210526,2.347826,7.666667,6.333333
2,2,dementia-005-2,He_Hinzen,MildAD,55,male,okay he's falling off a chair . she's running ...,0.222,0.723,0.055,...,4,19,0.000000,15.789474,21.052632,21.052632,1.333333,1.000000,inf,inf


In [41]:
# save the result to the original csv. file
merged_df3 = pd.merge(ad_clinical_speakers, df3[['id','Valence', 'Arousal', 'Dominance', 'SemD']],
                     on='id',
                     how='left')
merged_df3

,idx,id,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj,Valence,Arousal,Dominance,SemD
0,0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...,0.029,0.951,0.020,...,25.252525,10.101010,0.555556,2.500000,5.000000,9.000000,5.899231,3.999615,5.381538,2.103371
1,1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...,0.013,0.916,0.071,...,30.167598,12.849162,1.210526,2.347826,7.666667,6.333333,6.071000,3.676000,5.697200,2.155569
2,2,dementia-005-2,He_Hinzen,MildAD,55,male,okay he's falling off a chair . she's running ...,0.222,0.723,0.055,...,21.052632,21.052632,1.333333,1.000000,inf,inf,6.395000,3.863333,5.901667,2.037778
3,3,dementia-007-3,He_Hinzen,ModerateAD,75,female,well the girl is telling the boy to get the co...,0.054,0.893,0.053,...,23.456790,16.049383,0.812500,1.461538,inf,inf,6.067917,3.850417,5.549167,2.074805
4,4,dementia-010-0,He_Hinzen,MildAD,66,male,oh boy . wowie the boy's going up on a cookiej...,0.006,0.949,0.044,...,25.000000,16.346154,0.809524,1.529412,6.800000,8.400000,5.872742,3.706613,5.532742,2.044388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,190,Baycrest11976,Baycrest,MCI,88;00.,male,alright. you're going. uh the reason I'm laugh...,0.042,0.869,0.089,...,30.674157,10.786517,0.653061,2.843750,4.000000,6.125000,5.860737,4.013410,5.589447,2.128037
191,191,Baycrest12257,Baycrest,MCI,74;00.,female,right. oh gosh. I don't really know. I guess m...,0.024,0.898,0.078,...,23.643123,10.631970,0.674528,2.223776,5.500000,8.153846,6.005853,3.899197,5.714415,2.125885
192,192,Baycrest12813,Baycrest,MCI,66;00.,male,thinking back tell you a story. I have an inte...,0.050,0.844,0.105,...,30.235294,11.411765,0.636066,2.649485,4.619048,7.261905,5.882268,3.904943,5.627619,2.121081
193,193,Baycrest7352,Baycrest,MCI,71;00.,female,can you repeat the question? okay well as I si...,0.034,0.863,0.103,...,25.000000,16.292135,1.017544,1.534483,4.833333,4.750000,6.093936,3.927766,5.760532,2.111918


In [42]:
# save the results
merged_df3.to_csv(brief + 'AD_clinical_result.csv', index=False)

# Health (clinical label group)

In [43]:
#read the data
health_clinical = pd.read_csv(result + 'Health_clinical_lexicon.csv')
health_clinical.shape

(7108, 9)

In [44]:
health_clinical.head(3)

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,the,['the'],1,['the'],['PRON'],['DT']
1,0,control-002-0,the scene is in the kitchen . the mother is wi...,scene,['scene'],1,['scene'],['NOUN'],['NN']
2,0,control-002-0,the scene is in the kitchen . the mother is wi...,is,['is'],1,['be'],['AUX'],['VBZ']


In [45]:
# Apply the helper function to each row in the DataFrame
health_clinical[['Valence', 'Arousal', 'Dominance', 'SemD']] = health_clinical['word'].apply(get_emotion_values)

# Display the DataFrame
health_clinical

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,the,['the'],1,['the'],['PRON'],['DT'],NaN,NaN,NaN,2.37
1,0,control-002-0,the scene is in the kitchen . the mother is wi...,scene,['scene'],1,['scene'],['NOUN'],['NN'],5.94,1.95,5.29,1.91
2,0,control-002-0,the scene is in the kitchen . the mother is wi...,is,['is'],1,['be'],['AUX'],['VBZ'],NaN,NaN,NaN,2.37
3,0,control-002-0,the scene is in the kitchen . the mother is wi...,in,['in'],1,['in'],['ADP'],['IN'],NaN,NaN,NaN,2.37
4,0,control-002-0,the scene is in the kitchen . the mother is wi...,the,['the'],1,['the'],['PRON'],['DT'],NaN,NaN,NaN,2.37
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7103,103,99-1,I see a woman drying dishes with her sink over...,dishes,['dishes'],1,['dish'],['NOUN'],['NNS'],NaN,NaN,NaN,1.26
7104,103,99-1,I see a woman drying dishes with her sink over...,with,['with'],1,['with'],['ADP'],['IN'],NaN,NaN,NaN,2.37
7105,103,99-1,I see a woman drying dishes with her sink over...,her,['her'],1,['her'],['PRON'],['PRP$'],NaN,NaN,NaN,1.94
7106,103,99-1,I see a woman drying dishes with her sink over...,sink,['sink'],1,['sink'],['VERB'],['VB'],4.62,3.70,4.10,1.67


In [46]:
# save the results
health_clinical.to_csv(result + 'Health_clinical_lexicon.csv', index=False)
print("The file was successfully saved")

The file was successfully saved


In [47]:
df4 = health_clinical.groupby(['id']).agg({'Valence': lambda x: np.mean(x),
                                 'Arousal': lambda x: np.mean(x),
                                 'Dominance': lambda x: np.mean(x),
                                  'SemD': lambda x: np.mean(x)}).reset_index()
df4.head()

,id,Valence,Arousal,Dominance,SemD
0,02-1,5.34625,4.0500,5.15375,2.044348
1,03-1,5.56500,4.4325,5.49750,1.824000
2,04-1,NaN,NaN,NaN,1.590000
3,07-1,6.29000,3.9140,5.69200,2.034000
4,08-1,7.21500,4.3950,6.41000,1.882500


In [48]:
df4.columns

Index(['id', 'Valence', 'Arousal', 'Dominance', 'SemD'], dtype='object')

In [49]:
#load the file to save the result
health_clinical_speakers = pd.read_csv(result + 'Health_clinical_result.csv',engine='python')
health_clinical_speakers.head(3)

,idx,id,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...,0.010,0.925,0.064,...,26,124,6.451613,16.935484,20.967742,20.967742,1.238095,1.000000,3.25,2.625
1,1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...,0.020,0.980,0.000,...,16,78,0.000000,21.794872,23.076923,20.512821,0.941176,1.125000,inf,inf
2,2,control-015-1,He_Hinzen,Control,67,female,okay . a little boy is stepping on a stool tha...,0.029,0.923,0.048,...,33,152,3.289474,13.157895,22.368421,21.710526,1.650000,1.030303,6.60,4.000


In [50]:
# save the result to the original csv. file
merged_df4 = pd.merge(health_clinical_speakers, df4[['id','Valence', 'Arousal', 'Dominance', 'SemD']],
                     on='id',
                     how='left')
merged_df4

,idx,id,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj,Valence,Arousal,Dominance,SemD
0,0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...,0.010,0.925,0.064,...,20.967742,20.967742,1.238095,1.000000,3.250,2.625,6.400444,3.810444,5.840889,2.065556
1,1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...,0.020,0.980,0.000,...,23.076923,20.512821,0.941176,1.125000,inf,inf,6.124783,3.626522,5.844783,2.035921
2,2,control-015-1,He_Hinzen,Control,67,female,okay . a little boy is stepping on a stool tha...,0.029,0.923,0.048,...,22.368421,21.710526,1.650000,1.030303,6.600,4.000,5.857045,3.820455,5.553182,2.029214
3,3,control-017-4,He_Hinzen,Control,71,female,are you ready ? well the sink is overflowing ....,0.090,0.827,0.083,...,24.434389,14.932127,0.785714,1.636364,4.125,5.250,5.878267,3.947200,5.578800,2.088235
4,4,control-021-1,He_Hinzen,Control,74,female,okay . the mother's washing the dishes and the...,0.007,0.963,0.030,...,27.215190,15.189873,0.827586,1.791667,6.000,7.250,6.225238,3.853571,5.661667,2.110581
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,99,94-1,Delaware,Control,68;00.,female,yep,0.000,0.000,1.000,...,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.780000
100,100,96-1,Delaware,Control,61;00.,female,yep,0.000,0.000,1.000,...,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.780000
101,101,97-1,Delaware,Control,84;00.,female,yeah,0.000,0.000,1.000,...,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.180000
102,102,98-1,Delaware,Control,63;00.,female,okay,0.000,0.000,1.000,...,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.380000


In [51]:
# save the results
merged_df4.to_csv(brief + 'Health_clinical_result.csv', index=False)